# Introduction à Hugging Face 🤗

Ce notebook est un atelier pratique pour découvrir la bibliothèque Transformers de Hugging Face, qui permet d'utiliser facilement des modèles de traitement du langage naturel (NLP) à l'état de l'art.

## Objectifs du workshop:
- Comprendre l'utilisation des pipelines Hugging Face
- Explorer différents types de modèles (sentiment-analysis, text-generation, zero-shot-classification)
- Apprendre à manipuler les tokenizers
- Utiliser les modèles avec PyTorch pour plus de flexibilité

## Qu'est-ce que Hugging Face?

Hugging Face est une entreprise qui développe des outils pour construire, entraîner et déployer des modèles de machine learning, avec un focus particulier sur les modèles NLP. La bibliothèque `transformers` est leur produit phare, permettant d'accéder facilement à des milliers de modèles pré-entraînés.


## 1. Pipelines: la façon la plus simple d'utiliser les modèles

Les pipelines sont l'outil le plus simple pour utiliser les modèles pré-entraînés pour diverses tâches. Ils regroupent trois étapes:
1. Prétraitement (tokenization)
2. Passage des entrées à travers le modèle
3. Post-traitement des sorties du modèle

### 1.1 Analyse de sentiment

In [27]:
from transformers import pipeline

sentiment_classifier = pipeline("sentiment-analysis")

result = sentiment_classifier("I hate this movie")
print("Analyse de sentiment:")
print(result)

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


Analyse de sentiment:
[{'label': 'NEGATIVE', 'score': 0.9996687173843384}]


### 1.2 Génération de texte

La génération de texte permet de créer automatiquement du contenu textuel à partir d'une amorce.
Nous allons tester d'abord avec le modèle par défaut, puis avec un modèle spécifique.

### 1.3 Classification zero-shot

La classification zero-shot permet de classer un texte selon des catégories que le modèle n'a jamais vues pendant l'entraînement.

In [28]:
text_generator = pipeline("text-generation")

prompt = "My favorite color is"
result = text_generator(prompt)
print("\nGénération de texte (modèle par défaut):")
print(result)

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d (https://huggingface.co/openai-community/gpt2).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Génération de texte (modèle par défaut):
[{'generated_text': "My favorite color is the lilac shade, but I like it a lot more with rose and browns. I have only had one jar from the time I got it, so it's a little more than I am used to. The jar is"}]


In [29]:
specific_generator = pipeline("text-generation", model="distilgpt2")
results = specific_generator(
    prompt,
    max_length=30,
    num_return_sequences=2
)

print("\nGénération de texte (distilgpt2 avec paramètres):")
for i, result in enumerate(results):
    print(f"Séquence {i+1}: {result['generated_text']}")

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Génération de texte (distilgpt2 avec paramètres):
Séquence 1: My favorite color is dark but not too dark yet? No problem. I was really curious about how black felt after an entire day at work.

Séquence 2: My favorite color is White. This is a really great choice in case you're looking to add more color. You'll need to add a few bit


Création d'un pipeline de classification zero-shot

In [30]:
zero_shot_classifier = pipeline("zero-shot-classification")

text = "AI is rapidly transforming industries and society, raising both exciting opportunities and serious ethical concerns."

candidate_labels = ["artificial intelligence", "politics", "business"]

result = zero_shot_classifier(text, candidate_labels=candidate_labels)

print("\nClassification zero-shot:")
print(f"Texte: {text}")
print("Résultats:")
for label, score in zip(result["labels"], result["scores"]):
    print(f"- {label}: {score:.4f}")

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1 (https://huggingface.co/facebook/bart-large-mnli).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu



Classification zero-shot:
Texte: AI is rapidly transforming industries and society, raising both exciting opportunities and serious ethical concerns.
Résultats:
- artificial intelligence: 0.9767
- business: 0.0205
- politics: 0.0028



## 2. Comprendre les Tokenizers

Les tokenizers sont responsables de la conversion du texte en un format numérique que les modèles peuvent comprendre.
Le processus de tokenization comporte généralement les étapes suivantes:
1. Découper le texte en mots, sous-mots ou caractères (tokens)
2. Convertir ces tokens en identifiants numériques (token IDs)
3. Ajouter des tokens spéciaux comme [CLS], [SEP], [PAD] (je parlerai du token [CLS] en détail dans un prochain post 😉)
4. Créer des masques d'attention et autres entrées nécessaires au modèle

In [18]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [19]:
# Chargement d'un tokenizer pré-entraîné
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [31]:
input_text = "Using a transformer network is simple"

# Tokenization complète (en une seule étape)
encoded = tokenizer(input_text)
print("\nRésultat de la tokenization complète:")
print(encoded)


Résultat de la tokenization complète:
{'input_ids': [101, 2478, 1037, 10938, 2121, 2897, 2003, 3722, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [32]:
# Tokenization étape par étape pour mieux comprendre
print("\nTokenization étape par étape:")

# 1. Découper en tokens
tokens = tokenizer.tokenize(input_text)
print(f"Tokens: {tokens}")

# 2. Convertir en IDs
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"Token IDs: {token_ids}")

# 3. Reconvertir les IDs en texte (décoder)
decoded_text = tokenizer.decode(token_ids)
print(f"Texte décodé: {decoded_text}")


Tokenization étape par étape:
Tokens: ['using', 'a', 'transform', '##er', 'network', 'is', 'simple']
Token IDs: [2478, 1037, 10938, 2121, 2897, 2003, 3722]
Texte décodé: using a transformer network is simple


### Explication de la tokenization

Dans un modèle transformer comme DistilBERT, le texte d'entrée est d'abord divisé en tokens (mots ou sous-mots) à l'aide d'un tokenizer. Chaque token est ensuite converti en un ID numérique unique, qui correspond à une entrée dans le vocabulaire du modèle.

Le dictionnaire retourné par le tokenizer contient plusieurs éléments:
- `input_ids`: les IDs des tokens, y compris les tokens spéciaux
- `attention_mask`: indique au modèle quels tokens considérer (1) et lesquels ignorer (0, généralement utilisé pour le padding)
- `token_type_ids` (pour certains modèles): permet de distinguer différentes séquences dans une même entrée

Les tokens spéciaux comme `[CLS]` (début de phrase) et `[SEP]` (fin ou séparateur) sont automatiquement ajoutés par le tokenizer.




## 3. Utilisation des modèles

Nous pouvons charger un modèle pré-entraîné et l'utiliser avec un tokenizer pour effectuer des prédictions.
"""

from transformers import AutoModelForSequenceClassification



In [33]:
model = AutoModelForSequenceClassification.from_pretrained(model_name)

custom_classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# Test de la pipeline
result = custom_classifier("I hate this movie")
print("\nRésultat avec notre modèle personnalisé:")
print(result)

Device set to use cpu



Résultat avec notre modèle personnalisé:
[{'label': 'NEGATIVE', 'score': 0.9996687173843384}]


## 4. Intégration avec PyTorch

Pour plus de flexibilité, nous pouvons utiliser directement les modèles avec PyTorch.
Cela nous permet d'accéder aux sorties brutes du modèle et d'effectuer des opérations personnalisées.

In [23]:
import torch
import torch.nn.functional as F

In [24]:
X_train = [
    "I hate this movie",
    "This movie was amazing!",
    "Absolutely terrible, I walked out halfway.",
    "I loved every minute of it.",
    "It was okay, not the best but not the worst.",
    "I'm so glad I watched it.",
    "Nothing special, just average.",
    "Truly disappointing.",
    "Visually stunning and emotionally powerful.",
    "I regret spending money on this."
]

In [34]:
batch = tokenizer(
    X_train,
    padding=True,          # Ajouter du padding pour avoir des séquences de même longueur
    truncation=True,       # Tronquer les séquences trop longues
    max_length=512,        # Longueur maximale des séquences
    return_tensors="pt"    # Retourner des tensors PyTorch
)

print("\nBatch d'entrée tokenisé:")
for key, value in batch.items():
    print(f"{key}: shape {value.shape}")

with torch.no_grad(): # obligatoire quand on ne fait pas de l'entrainement ;)
    outputs = model(**batch)

    print("\nSorties brutes du modèle:")
    print(f"Logits shape: {outputs.logits.shape}")

    predictions = F.softmax(outputs.logits, dim=1)
    print("\nProbabilités après softmax:")
    print(predictions)

    labels = torch.argmax(predictions, dim=1)
    print("\nLabels prédits (0: négatif, 1: positif):")
    print(labels)

    print("\nRésultats détaillés:")
    for i, (text, pred, label) in enumerate(zip(X_train, predictions, labels)):
        sentiment = "positif" if label == 1 else "négatif"
        confidence = pred[label].item()
        print(f"{i+1}. \"{text}\" - {sentiment} (confiance: {confidence:.4f})")



Batch d'entrée tokenisé:
input_ids: shape torch.Size([10, 14])
attention_mask: shape torch.Size([10, 14])

Sorties brutes du modèle:
Logits shape: torch.Size([10, 2])

Probabilités après softmax:
tensor([[9.9967e-01, 3.3127e-04],
        [1.1988e-04, 9.9988e-01],
        [9.9971e-01, 2.9101e-04],
        [1.3912e-04, 9.9986e-01],
        [1.1393e-02, 9.8861e-01],
        [7.7059e-04, 9.9923e-01],
        [9.9776e-01, 2.2433e-03],
        [9.9981e-01, 1.8757e-04],
        [1.1775e-04, 9.9988e-01],
        [9.9874e-01, 1.2586e-03]])

Labels prédits (0: négatif, 1: positif):
tensor([0, 1, 0, 1, 1, 1, 0, 0, 1, 0])

Résultats détaillés:
1. "I hate this movie" - négatif (confiance: 0.9997)
2. "This movie was amazing!" - positif (confiance: 0.9999)
3. "Absolutely terrible, I walked out halfway." - négatif (confiance: 0.9997)
4. "I loved every minute of it." - positif (confiance: 0.9999)
5. "It was okay, not the best but not the worst." - positif (confiance: 0.9886)
6. "I'm so glad I watched 

## Conclusion

Dans ce workshop, nous avons exploré les fonctionnalités de base de la bibliothèque Transformers de Hugging Face:

1. Utilisation des pipelines pour différentes tâches NLP
2. Compréhension du processus de tokenization
3. Chargement et utilisation de modèles pré-entraînés
4. Intégration avec PyTorch pour un contrôle plus fin

Ces outils permettent d'accéder facilement à des modèles de pointe pour le traitement du langage naturel sans avoir à les entraîner soi-même, ce qui rend l'IA accessible à un plus grand nombre de développeurs et de chercheurs.

### Pour aller plus loin:
- Explorer d'autres types de pipelines (traduction, résumé, question-réponse...)
- Fine-tuner des modèles sur des données spécifiques
- Utiliser des techniques d'optimisation pour déployer les modèles en production
- Explorer le Hugging Face Hub pour découvrir des milliers d'autres modèles